# Weather Pattern Clustering for Localised Climate Insight Generation
**Course:** INT396 (Unsupervised Machine Learning)  
**Problem Statement ID:** 31  
**Student Name:** Ganta Gangadhar  
**Registration Number:** 12523695  
**GitHub Repository:** [https://github.com/GangadharGanta/weather-pattern-clustering](https://github.com/GangadharGanta/weather-pattern-clustering)  
**Live Deployed App:** [https://weather-pattern-clustering.streamlit.app](https://weather-pattern-clustering.streamlit.app)  

---
### Abstract & Academic Purpose
This notebook provides a comprehensive, end-to-end unsupervised machine learning workflow to discover, segment, and evaluate localized climate patterns from continuous meteorological telemetry. Using **Temperature**, **Humidity**, **Wind Speed**, **Pressure**, and **Precipitation**, we evaluate K-Means, Agglomerative Hierarchical, and DBSCAN algorithms. Distance-sensitive feature standardization (`StandardScaler`) and dimensionality reduction (`PCA`) are formulated rigorously to achieve optimal cluster separation.

In [ ]:
# Environment setup and library imports
import os
import sys
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Machine Learning & Clustering Imports
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

# Visualization aesthetics
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.dpi'] = 150
print("All core scientific libraries loaded successfully!")

## 1. Data Ingestion & Preprocessing Pipeline
Weather measurements are continuous time-series. We ingest 1,826 daily records (5 complete seasonal cycles), handle missing entries through chronological forward-fill/backward-fill, enforce physical meteorological boundaries, and compute thermodynamic variables (Dew Point and Apparent Temperature).

In [ ]:
# Add root directory to sys.path to access modular src scripts
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.data_loader import load_raw_data, clean_weather_data, CORE_METEOROLOGICAL_FEATURES

raw_path = project_root / "data" / "raw" / "historical_weather_daily.csv"
raw_df = load_raw_data(raw_path)
df = clean_weather_data(raw_df)

print(f"Dataset Dimensions: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

In [ ]:
# Dataset Summary Statistics
df[CORE_METEOROLOGICAL_FEATURES].describe().T

## 2. Exploratory Data Analysis (EDA)
Understanding the physical and statistical relationships among atmospheric features:
- **Clausius-Clapeyron Moisture Dynamics:** Temperature governs maximum saturation vapor pressure.
- **Barometric Cyclonic Lows:** Strong inverse coupling between atmospheric pressure and precipitation/wind speed.
- **Annual Seasonality:** Cyclic temperature oscillation peaking during summer and dipping in winter.

In [ ]:
# Correlation Heatmap across Meteorological Features
plt.figure(figsize=(8, 6))
corr = df[CORE_METEOROLOGICAL_FEATURES].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt=".3f", cmap="coolwarm", mask=mask, cbar_kws={'label': 'Pearson Correlation'})
plt.title("Meteorological Correlation Matrix (Pearson)", fontsize=12, weight='bold', pad=10)
plt.show()

In [ ]:
# Multi-panel Seasonal Time-Series Trends
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
df_plot = df.set_index(pd.to_datetime(df['Date']))

axes[0].plot(df_plot['Temperature'], color='#e74c3c', lw=1.2, label='Temperature (°C)')
axes[0].set_ylabel('Temp (°C)', weight='bold')
axes[0].legend(loc='upper right')

axes[1].plot(df_plot['Humidity'], color='#3498db', lw=1.2, label='Humidity (%)')
axes[1].set_ylabel('Humidity (%)', weight='bold')
axes[1].legend(loc='upper right')

axes[2].plot(df_plot['Pressure'], color='#8e44ad', lw=1.2, label='Pressure (hPa)')
axes[2].set_ylabel('Pressure (hPa)', weight='bold')
axes[2].legend(loc='upper right')

axes[3].bar(df_plot.index, df_plot['Precipitation'], color='#16a085', width=1.5, label='Precipitation (mm)')
axes[3].set_ylabel('Rainfall (mm)', weight='bold')
axes[3].legend(loc='upper right')

plt.suptitle('Multi-Year Historical Meteorological Cycles (2020 - 2025)', fontsize=14, weight='bold', y=0.92)
plt.show()

## 3. Distance-Sensitive Feature Standardization (`StandardScaler`)
### Mathematical Imperative for Scaling
Clustering algorithms rely on Euclidean distances:
$$d(\mathbf{x}_i, \mathbf{x}_j) = \sqrt{\sum_{d=1}^D (x_{i,d} - x_{j,d})^2}$$
Atmospheric pressure ($pprox 1013\text{ hPa}$) is 100-fold larger than rainfall ($pprox 5\text{ mm}$) or wind speed ($pprox 15\text{ km/h}$). Without z-score standardization ($z = \frac{x - \mu}{\sigma}$), pressure would dominate distance calculations by over $99\%$. We apply `StandardScaler` to ensure zero mean ($\mu = 0$) and unit variance ($\sigma = 1$).

In [ ]:
# Feature matrix extraction and standardization
X_raw = df[CORE_METEOROLOGICAL_FEATURES].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

print("Standardized Feature Means (should be ~ 0.0):")
print(X_scaled.mean(axis=0).round(6))
print("Standardized Feature Standard Deviations (should be 1.0):")
print(X_scaled.std(axis=0).round(6))

## 4. Dimensionality Reduction via Principal Component Analysis (PCA)
PCA projects the 5-dimensional standardized space onto orthogonal eigenvectors that maximize captured atmospheric variance. This creates a clean 2D manifold for visual inspection.

In [ ]:
# Fit PCA with 2 orthogonal components
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

exp_var = pca.explained_variance_ratio_
print(f"PC1 Explained Variance: {exp_var[0]:.2%}")
print(f"PC2 Explained Variance: {exp_var[1]:.2%}")
print(f"Total 2D Explained Variance: {exp_var.sum():.2%}")

# Component Factor Loadings
loadings = pd.DataFrame(
    pca.components_.T,
    columns=['PC1', 'PC2'],
    index=CORE_METEOROLOGICAL_FEATURES
)
print("\nPCA Component Factor Loadings:")
display(loadings)

## 5. Hyperparameter Tuning: Elbow Method & Silhouette Analysis
We evaluate cluster candidates across $k \in [2, 8]$ using:
1. **Within-Cluster Sum of Squares (WCSS / Inertia):** Measures internal tightness.
2. **Silhouette Coefficient ($s$):** Evaluates cohesion versus separation ($s \in [-1, 1]$).
3. **Davies-Bouldin Index ($DB$):** Measures cluster similarity; lower values indicate superior separation.

In [ ]:
# Systematic hyperparameter evaluation across k in [2, 8]
ks = list(range(2, 9))
wcss_scores = []
sil_scores = []
db_scores = []
ch_scores = []

for k in ks:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    labels = km.fit_predict(X_scaled)
    wcss_scores.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))
    db_scores.append(davies_bouldin_score(X_scaled, labels))
    ch_scores.append(calinski_harabasz_score(X_scaled, labels))

tuning_df = pd.DataFrame({
    'k': ks,
    'WCSS (Inertia)': np.round(wcss_scores, 2),
    'Silhouette Score': np.round(sil_scores, 4),
    'Davies-Bouldin': np.round(db_scores, 4),
    'Calinski-Harabasz': np.round(ch_scores, 2)
})
display(tuning_df)

In [ ]:
# Plotting Dual-Axis Elbow and Silhouette Optimization Curve
fig, ax1 = plt.subplots(figsize=(10, 5))

color_wcss = '#2980b9'
color_sil = '#e67e22'

ax1.set_xlabel('Number of Clusters (k)', fontsize=11, weight='bold')
ax1.set_ylabel('Inertia / WCSS', color=color_wcss, fontsize=11, weight='bold')
l1 = ax1.plot(ks, wcss_scores, marker='o', color=color_wcss, lw=2.2, label='WCSS (Elbow)')
ax1.tick_params(axis='y', labelcolor=color_wcss)
ax1.grid(True, linestyle='--', alpha=0.4)

ax2 = ax1.twinx()
ax2.set_ylabel('Silhouette Score', color=color_sil, fontsize=11, weight='bold')
l2 = ax2.plot(ks, sil_scores, marker='s', color=color_sil, lw=2.2, ls='--', label='Silhouette Score')
ax2.tick_params(axis='y', labelcolor=color_sil)

# Mark optimal k=5
ax2.scatter([5], [sil_scores[3]], color='#c0392b', s=160, zorder=5, edgecolors='black')
ax2.annotate('Optimal k=5', xy=(5, sil_scores[3]), xytext=(5.3, sil_scores[3]+0.02),
             arrowprops=dict(facecolor='black', shrink=0.08, width=1.5, headwidth=6),
             fontweight='bold', color='#c0392b')

lines = l1 + l2
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='upper center')
plt.title('Elbow Method and Silhouette Analysis: Optimal k = 5', fontsize=13, weight='bold')
plt.show()

## 6. Multi-Algorithm Benchmark (K-Means vs Agglomerative vs DBSCAN)
We benchmark K-Means ($k=5$) against Agglomerative Hierarchical Clustering (Ward linkage) and DBSCAN (Density-Based Spatial Clustering).

In [ ]:
# 1. K-Means++
km_champion = KMeans(n_clusters=5, init='k-means++', n_init=10, random_state=42)
km_labels = km_champion.fit_predict(X_scaled)
km_sil = silhouette_score(X_scaled, km_labels)

# 2. Agglomerative Hierarchical Clustering
agg = AgglomerativeClustering(n_clusters=5, linkage='ward')
agg_labels = agg.fit_predict(X_scaled)
agg_sil = silhouette_score(X_scaled, agg_labels)

# 3. DBSCAN
db = DBSCAN(eps=1.1, min_samples=15)
db_labels = db.fit_predict(X_scaled)
n_db_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
noise_ratio = np.sum(db_labels == -1) / len(db_labels)

comparison_df = pd.DataFrame([
    {'Algorithm': 'K-Means++', 'Clusters': 5, 'Silhouette Score': round(km_sil, 4), 'Inference Latency': '<1 ms', 'Noise Points': 0},
    {'Algorithm': 'Agglomerative (Ward)', 'Clusters': 5, 'Silhouette Score': round(agg_sil, 4), 'Inference Latency': 'O(N) search', 'Noise Points': 0},
    {'Algorithm': 'DBSCAN', 'Clusters': n_db_clusters, 'Silhouette Score': 'N/A', 'Inference Latency': 'Density query', 'Noise Points': f"{noise_ratio:.1%}"}
])
print("Algorithm Comparison Table:")
display(comparison_df)

## 7. Latent Space Visualization & Cluster Profiling
Projecting the clusters onto the 2D PCA plane and inspecting their unscaled centroid values to interpret localized synoptic weather regimes.

In [ ]:
# Plot 2D PCA Cluster Scatter Plot
palette = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']
cluster_names = [
    'C0: Hot & Humid (Monsoon)',
    'C1: Cold & Dry (Winter)',
    'C2: Stormy & Unstable (Gale)',
    'C3: Hot & Arid (Heatwave)',
    'C4: Pleasant & Temperate (Spring)'
]

plt.figure(figsize=(10, 7))
for c_id in range(5):
    mask = km_labels == c_id
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1], c=palette[c_id], label=cluster_names[c_id], alpha=0.55, s=35)

# Centroids in PCA space
centroids_pca = pca.transform(km_champion.cluster_centers_)
plt.scatter(centroids_pca[:, 0], centroids_pca[:, 1], c='black', marker='X', s=200, edgecolors='white', label='Centroids', zorder=10)

plt.title('2D PCA Projection of Discovered Weather Regimes', fontsize=14, weight='bold', pad=12)
plt.xlabel(f'Principal Component 1 ({exp_var[0]:.1%} Variance)', weight='bold')
plt.ylabel(f'Principal Component 2 ({exp_var[1]:.1%} Variance)', weight='bold')
plt.legend(frameon=True, loc='upper right')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

In [ ]:
# Unscaled Cluster Centroid Summary Table
df['Cluster'] = km_labels
centroids_unscaled = scaler.inverse_transform(km_champion.cluster_centers_)
centroid_df = pd.DataFrame(centroids_unscaled, columns=CORE_METEOROLOGICAL_FEATURES)
centroid_df['Cluster'] = range(5)
centroid_df['Regime Name'] = [c.split(':')[1].strip() for c in cluster_names]
centroid_df['Observation Count'] = [np.sum(km_labels == i) for i in range(5)]
centroid_df['Frequency (%)'] = [round(np.sum(km_labels == i) / len(km_labels) * 100, 2) for i in range(5)]

display(centroid_df[['Cluster', 'Regime Name', 'Observation Count', 'Frequency (%)'] + CORE_METEOROLOGICAL_FEATURES].round(2))

## 8. Champion Model Export & Artifact Serialization
We serialize the complete champion pipeline (`StandardScaler`, `PCA`, and `KMeans`) to `models/kmeans_champion.joblib` and save the cluster profile metadata to `models/cluster_profiles.json` for live Streamlit web deployment.

In [ ]:
from src.train import train_and_select_champion

metrics_out = train_and_select_champion(
    data_path=str(project_root / "data" / "processed" / "processed_weather_data.csv"),
    output_model_path=str(project_root / "models" / "kmeans_champion.joblib"),
    output_profiles_path=str(project_root / "models" / "cluster_profiles.json"),
    optimal_k=5
)
print("Training Complete! Metrics summary:")
print(json.dumps(metrics_out, indent=2))